# 06 — Train Conditional DCGAN (Phase 7)

**Goal:** Train a class-conditioned DCGAN on ISIC 2018 Task 3 so Phase 8 can sample synthetic minority-class images. This is the *sharp* counterpart to the Phase 6 CVAE — VAEs are stable but blurry, GANs are unstable but sharp.

**Reference course material:** `notes13gan.pdf` (DCGAN G/D architecture, training procedure, BCE loss), `notes12generativeAI.pdf` (cGAN concept).

**Key design choices** (documented in `src/models/cgan.py`):
- DCGAN (Radford et al. 2016): ConvTranspose+BN+LeakyReLU in G, Conv+BN+LeakyReLU in D, Tanh at G output, Sigmoid at D output, weights init from N(0, 0.02).
- **Conditioning**: one-hot label concatenated to noise `z` (G); spatially broadcast to 7 extra channels and concatenated to the input image (D).
- Output range **[-1, 1]** because of Tanh — so the training transform normalises with mean=0.5, std=0.5 (NOT ImageNet, NOT [0,1] like the CVAE).
- Adam, **lr=2e-4, betas=(0.5, 0.999)** — DCGAN-paper defaults.
- **No EarlyStopping.** GANs don't have a meaningful val loss — the losses bounce around as G and D play each other. We train for a fixed number of epochs, snapshot every few epochs, and pick the best by visual inspection and (in Phase 9) FID.

**Outputs** (matching the Phase 5/6 convention):
- `results/checkpoints/cgan_G_final.pt`, `cgan_D_final.pt`
- `results/checkpoints/cgan_G_epochXX.pt` (periodic snapshots, so we can pick the best one later)
- `results/logs/cgan_log.csv`
- `results/plots/cgan_losses.png`
- `results/plots/cgan_samples_epochXX.png` (fixed-noise sample grid per snapshot)
- `results/plots/cgan_samples_per_class_final.png`

## 1. Setup, paths, seed

In [ ]:
import sys
from pathlib import Path
import csv
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import ISICDataset, CLASS_NAMES          # noqa: E402
from src.models.cgan import (                              # noqa: E402
    Generator, Discriminator,
    init_dcgan_weights, sample_images,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')

RESULTS = PROJECT_ROOT / 'results'
(RESULTS / 'checkpoints').mkdir(parents=True, exist_ok=True)
(RESULTS / 'logs').mkdir(parents=True, exist_ok=True)
(RESULTS / 'plots').mkdir(parents=True, exist_ok=True)

LOG_PATH = RESULTS / 'logs' / 'cgan_log.csv'
print(f'Log -> {LOG_PATH}')

## 2. Hyperparameters

> **Why fewer epochs than the CVAE?** GANs see *each batch twice per step* (once for D, once for G), and instability gets worse the longer you train. 50 epochs is a sensible starting point at 224×224; if the snapshots are still improving at the end, raise this.

In [ ]:
IMG_SIZE       = 224
BATCH_SIZE     = 32       # drop to 16 if OOM (GANs are memory-hungry)
LATENT_DIM     = 100      # DCGAN paper default
N_CLASSES      = len(CLASS_NAMES)   # 7
EPOCHS         = 50
LR             = 2e-4     # DCGAN paper default
BETAS          = (0.5, 0.999)   # DCGAN paper default
SNAPSHOT_EVERY = 5        # save G/D + sample grid every N epochs
NUM_WORKERS    = 4

print(f'image size     : {IMG_SIZE}')
print(f'batch size     : {BATCH_SIZE}')
print(f'latent dim     : {LATENT_DIM}')
print(f'epochs         : {EPOCHS}')
print(f'lr             : {LR}    betas={BETAS}')
print(f'snapshot every : {SNAPSHOT_EVERY} epochs')

## 3. Data — Tanh-friendly transforms ([-1, 1] range)

Same `ISICDataset` and same `splits.csv` as Phases 5 and 6, but the transform normalises to **[-1, 1]** (mean=0.5, std=0.5 per channel) to match the Generator's Tanh output.

We only use the `train` split — GAN training has no validation: there's nothing to validate against, since the loss is adversarial.

In [ ]:
# [0,1]  --(mean=0.5, std=0.5)-->  [-1, 1]
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),                             # [0, 1]
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),  # [-1, 1]
])

splits_csv = PROJECT_ROOT / 'data' / 'processed' / 'splits.csv'
images_dir = PROJECT_ROOT / 'data' / 'raw' / 'ISIC2018_Task3_Training_Input'

train_ds = ISICDataset(splits_csv, images_dir, split='train', transform=train_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

print(f'train batches : {len(train_loader)}   ({len(train_ds)} images)')

x, y = next(iter(train_loader))
print(f'\nbatch x : {tuple(x.shape)}   dtype={x.dtype}   '
      f'min={x.min():.3f}  max={x.max():.3f}   (expected ~[-1, 1])')
print(f'batch y : {tuple(y.shape)}   unique={sorted(y.unique().tolist())}')

## 4. Model, optimizers, loss

In [ ]:
G = Generator(latent_dim=LATENT_DIM, n_classes=N_CLASSES).to(DEVICE)
D = Discriminator(n_classes=N_CLASSES).to(DEVICE)
G.apply(init_dcgan_weights)
D.apply(init_dcgan_weights)

opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=BETAS)

criterion = nn.BCELoss()   # matches notes13gan.pdf

print(f'Generator     params: {sum(p.numel() for p in G.parameters()):,}')
print(f'Discriminator params: {sum(p.numel() for p in D.parameters()):,}')

# A fixed noise+label grid we'll use for monitoring -- 4 samples per class,
# regenerated at every snapshot epoch from the SAME z and y so we can see
# how that fixed point in latent space evolves over training.
MONITOR_N = 4
fixed_z = torch.randn(MONITOR_N * N_CLASSES, LATENT_DIM, device=DEVICE)
fixed_y = torch.arange(N_CLASSES, device=DEVICE).repeat_interleave(MONITOR_N)
print(f'\nfixed monitor batch: {fixed_z.shape[0]} samples '
      f'({MONITOR_N} per class)')

## 5. Snapshot helper — save G/D and a sample grid

In [ ]:
def save_snapshot(epoch: int) -> None:
    """Save G, D, and a (n_classes x MONITOR_N) grid of fixed-noise samples."""
    g_path = RESULTS / 'checkpoints' / f'cgan_G_epoch{epoch:02d}.pt'
    d_path = RESULTS / 'checkpoints' / f'cgan_D_epoch{epoch:02d}.pt'
    torch.save({
        'epoch': epoch, 'state_dict': G.state_dict(),
        'latent_dim': LATENT_DIM, 'n_classes': N_CLASSES,
    }, g_path)
    torch.save({
        'epoch': epoch, 'state_dict': D.state_dict(),
        'n_classes': N_CLASSES,
    }, d_path)

    G.eval()
    with torch.no_grad():
        imgs = G(fixed_z, fixed_y)                # (n_classes*MONITOR_N, 3, 224, 224) in [-1,1]
        imgs = ((imgs + 1.0) / 2.0).clamp(0.0, 1.0)
    G.train()

    imgs_np = imgs.cpu().permute(0, 2, 3, 1).numpy()
    fig, axes = plt.subplots(N_CLASSES, MONITOR_N,
                             figsize=(1.8 * MONITOR_N, 1.8 * N_CLASSES))
    for c in range(N_CLASSES):
        for j in range(MONITOR_N):
            ax = axes[c, j]
            ax.imshow(imgs_np[c * MONITOR_N + j])
            ax.axis('off')
            if j == 0:
                ax.set_ylabel(CLASS_NAMES[c], fontsize=10, rotation=0,
                              labelpad=28, va='center')
    plt.suptitle(f'cGAN samples — epoch {epoch} (fixed z)', y=1.01)
    plt.tight_layout()
    out = RESULTS / 'plots' / f'cgan_samples_epoch{epoch:02d}.png'
    plt.savefig(out, dpi=110, bbox_inches='tight')
    plt.close(fig)
    print(f'  -> saved snapshot: {g_path.name}, {out.name}')

## 6. Training loop

Per batch:
1. **Train D** on a real batch (target=1) and a fake batch (target=0). Backprop, step D.
2. **Train G** by generating a new fake batch and computing BCE against target=1 (we want D to think these are real). Backprop *through D* but only step G.

Both steps use the **class label** to condition. The label for the fake batch when training D is the same one the generator was conditioned on — that's how D learns class consistency, not just realism.

**What to watch:**
- `D_loss` and `G_loss` typically oscillate. That's normal.
- `D(x)` ≈ probability D assigns to real samples — should hover around 0.5–0.8.
- `D(G(z))` before G step ≈ probability D assigns to fakes — should hover around 0.2–0.5.
- **If `D_loss → 0` and `D(G(z)) → 0`**: D won. G stopped learning. Lower D's lr or train G twice per D step.
- **If `G_loss` explodes upward**: mode collapse coming. Look at the snapshots.

In [ ]:
with open(LOG_PATH, 'w', newline='') as f:
    csv.writer(f).writerow(['epoch', 'D_loss', 'G_loss', 'D_x', 'D_G_z1', 'D_G_z2'])

history = {'epoch': [], 'D_loss': [], 'G_loss': [],
           'D_x': [], 'D_G_z1': [], 'D_G_z2': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    G.train(); D.train()

    run = {'D_loss': 0.0, 'G_loss': 0.0, 'D_x': 0.0, 'D_G_z1': 0.0, 'D_G_z2': 0.0}
    n_batches = 0

    for real, y in train_loader:
        bsz = real.size(0)
        real = real.to(DEVICE, non_blocking=True)
        y    = y.to(DEVICE, non_blocking=True)

        ones  = torch.ones (bsz, 1, device=DEVICE)
        zeros = torch.zeros(bsz, 1, device=DEVICE)

        # ============================================================
        # 1)  Train D : maximise log D(x, y) + log(1 - D(G(z, y), y))
        # ============================================================
        opt_D.zero_grad()

        # real batch
        d_real = D(real, y)
        loss_d_real = criterion(d_real, ones)

        # fake batch  (detach so G isn't updated here)
        z = torch.randn(bsz, LATENT_DIM, device=DEVICE)
        fake = G(z, y)
        d_fake = D(fake.detach(), y)
        loss_d_fake = criterion(d_fake, zeros)

        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        opt_D.step()

        # ============================================================
        # 2)  Train G : maximise log D(G(z, y), y)  -- target = 1
        # ============================================================
        opt_G.zero_grad()
        # Re-run D on the SAME fake batch (no detach this time)
        d_fake_for_g = D(fake, y)
        loss_g = criterion(d_fake_for_g, ones)
        loss_g.backward()
        opt_G.step()

        # bookkeeping
        run['D_loss'] += loss_d.item()
        run['G_loss'] += loss_g.item()
        run['D_x']    += d_real.mean().item()
        run['D_G_z1'] += d_fake.mean().item()           # before G step
        run['D_G_z2'] += d_fake_for_g.mean().item()     # after  G step
        n_batches += 1

    # per-epoch averages
    for k in run:
        run[k] /= n_batches

    history['epoch'].append(epoch)
    for k, v in run.items():
        history[k].append(v)

    with open(LOG_PATH, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, run['D_loss'], run['G_loss'],
                                run['D_x'], run['D_G_z1'], run['D_G_z2']])

    dt = time.time() - t0
    print(f'Epoch {epoch:3d}/{EPOCHS}  '
          f"D_loss={run['D_loss']:.3f}  G_loss={run['G_loss']:.3f}  |  "
          f"D(x)={run['D_x']:.3f}  D(G(z))={run['D_G_z1']:.3f}->{run['D_G_z2']:.3f}  "
          f"({dt:.1f}s)")

    if epoch % SNAPSHOT_EVERY == 0 or epoch == EPOCHS:
        save_snapshot(epoch)

# Always save a final, named checkpoint -- this is what Phase 8 loads
torch.save({
    'epoch': EPOCHS, 'state_dict': G.state_dict(),
    'latent_dim': LATENT_DIM, 'n_classes': N_CLASSES,
}, RESULTS / 'checkpoints' / 'cgan_G_final.pt')
torch.save({
    'epoch': EPOCHS, 'state_dict': D.state_dict(),
    'n_classes': N_CLASSES,
}, RESULTS / 'checkpoints' / 'cgan_D_final.pt')
print('\nTraining done. Saved cgan_G_final.pt and cgan_D_final.pt')

## 7. Plot training curves

Different shape than VAE curves: there's no "converging val loss." What we want to see is **D and G losses that oscillate around each other** — that means the game is balanced. If one flatlines, the other won.

In [ ]:
log = pd.read_csv(LOG_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(log['epoch'], log['D_loss'], label='D loss')
axes[0].plot(log['epoch'], log['G_loss'], label='G loss')
axes[0].set_title('Adversarial losses')
axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(log['epoch'], log['D_x'],    label='D(x)        real')
axes[1].plot(log['epoch'], log['D_G_z1'], label='D(G(z)) pre-G step')
axes[1].plot(log['epoch'], log['D_G_z2'], label='D(G(z)) post-G step')
axes[1].axhline(0.5, color='k', linestyle=':', alpha=0.5, label='0.5 = perfect equilibrium')
axes[1].set_ylim(0, 1)
axes[1].set_title('Discriminator confidences')
axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
out = RESULTS / 'plots' / 'cgan_losses.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()

## 8. Final per-class sample grid

6 fresh samples per class from `G_final`. This is what Phase 8 will use to augment the minority classes.

> If a row looks identical across all 6 samples → **mode collapse** for that class. Common fixes: train longer, lower D's lr, add label smoothing (real → 0.9). If it happens on rare classes only, that's expected — D barely sees them. We compensate in Phase 8 by mixing in CVAE samples for those classes.

In [ ]:
N_PER_CLASS = 6
fig, axes = plt.subplots(N_CLASSES, N_PER_CLASS,
                         figsize=(1.8 * N_PER_CLASS, 1.8 * N_CLASSES))

for c in range(N_CLASSES):
    samples = sample_images(G, class_idx=c, n=N_PER_CLASS, device=DEVICE)
    samples = samples.cpu().permute(0, 2, 3, 1).numpy()
    for j in range(N_PER_CLASS):
        ax = axes[c, j]
        ax.imshow(samples[j])
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(CLASS_NAMES[c], fontsize=11, rotation=0,
                          labelpad=30, va='center')

plt.suptitle('cGAN samples — 6 per class from final G', y=1.01)
plt.tight_layout()
out = RESULTS / 'plots' / 'cgan_samples_per_class_final.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()

## 9. What to look at after running

1. **`cgan_losses.png`**: D_loss and G_loss should *oscillate around each other*, not diverge. D(x) ideally stays around 0.6–0.8, D(G(z)) around 0.2–0.4.
2. **Snapshots `cgan_samples_epochXX.png`**: scroll through them — pick the epoch where samples look best by eye. That's our "best G," not necessarily the last epoch. Save its checkpoint path; Phase 9 (FID) will give an objective second opinion.
3. **Mode collapse check**: in the final per-class grid, every row should show variation. Identical samples = collapse for that class.
4. **Reality check**: skin-lesion GANs at 224×224 from only ~7k images are *hard*. Expect rough, lo-fi but recognisable samples — not photorealistic skin. If samples are pure noise after ~20 epochs, something's wrong (most likely D won; lower LR_D or train G twice per step).

**When happy**, update PROJECT_STATUS.md: mark Phase 7 ✅, note the best snapshot epoch, and which classes had mode collapse (if any). Phase 8 is next — it loads `cgan_G_final.pt` (or whichever snapshot won) and `cvae_best.pt`, then dumps synthetic minority-class images to disk for Phase 10.